# MRAG Ablation Study — Data Collection

Runs six leave-one-out ablations against the 150-question MUTCD-150 gold set.
Captures raw RAG output only. **No scoring, no GPT calls, no interpretation.**

Each ablation is a **self-contained cell** — if A3 fails, re-run only cell A3.
Each cell packages its output into a zip on Drive at `RESULTS_DRIVE_DIR/mrag_ablation_<id>.zip`.

Hand each zip to GPT-5.6 for judgment. This notebook does not.

## Cell order
1. Environment setup (mount + clone)
2. Install deps (VLM providers only — no openai)
3. API keys
4. Configuration — set paths + `RESULTS_DRIVE_DIR`
5. Sanity check the gold file
6. Pipeline init (once per session)
7. Ablation list + helper definition
8. Smoke test (optional, 5 Qs on A1)
9–14. **Per-ablation cells A1–A6** — each self-contained, run in any order
15. Run all remaining (convenience loop)
16. Handoff notes


## 1. Environment setup

In [ ]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
print('Colab detected:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    else:
        print('Drive already mounted')


In [ ]:
REPO_URL = 'https://github.com/hannanazad/MRAG_stp2.git'
REPO_DIR = '/content/MRAG_stp2'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Repo ready at', REPO_DIR)


## 2. Install deps (VLM providers only — no openai)

In [ ]:
# Only what the ablation harness itself needs beyond what the main mrag/
# requirements already provide. NO openai — GPT scoring happens elsewhere.
!pip install -q pyyaml


## 3. API keys (VLM providers only)

The RAG needs whichever provider your VLM alias routes to. `fable` = Anthropic → needs `ANTHROPIC_API_KEY`.

In [ ]:
# Load VLM provider keys from Colab Secrets. NOT loading OPENAI — this
# notebook does not call GPT.
try:
    from google.colab import userdata
    for k in ('DASHSCOPE_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY', 'QWEN'):
        try:
            v = userdata.get(k)
            if v:
                # Legacy "QWEN" secret maps to DASHSCOPE_API_KEY per your KG v4.6 fix.
                target = 'DASHSCOPE_API_KEY' if k == 'QWEN' else k
                os.environ[target] = v
                print(f'  {k} -> {target}: loaded from Colab Secrets')
        except Exception:
            pass
except ImportError:
    print('Not in Colab; expecting env vars already set.')

print()
print('Env check:',
      'DASHSCOPE_API_KEY set:', bool(os.environ.get('DASHSCOPE_API_KEY')),
      ' ANTHROPIC_API_KEY set:', bool(os.environ.get('ANTHROPIC_API_KEY')),
      ' GEMINI_API_KEY set:', bool(os.environ.get('GEMINI_API_KEY')))


## 4. Configuration

Edit these paths, then run. `RESULTS_DRIVE_DIR` is where the per-ablation zips will land.

In [ ]:
from pathlib import Path

# --- CHANGE THESE ---
GOLD_QA_PATH        = '/content/drive/MyDrive/MRAG/eval/gold_qa.jsonl'
RESULTS_DRIVE_DIR   = '/content/drive/MyDrive/MRAG/eval/ablations'

# Model + prompt style — frozen for the study.
VLM_ALIAS    = 'fable'
PROMPT_STYLE = 'fewshot'

# Cheap first pass: retrieval only (no generation). Minutes per ablation.
# Set False for the full generation sweep (~70 min per ablation).
RETRIEVAL_ONLY = False

# Limit for debugging (None = all 150).
LIMIT_N_QUESTIONS = None

# --- computed paths ---
RESULTS_DRIVE_DIR = Path(RESULTS_DRIVE_DIR)
RESULTS_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# Local scratch for jsonl files before zipping to Drive.
LOCAL_RESULTS_DIR = Path('/content/ablation_results')
LOCAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Local scratch dir :', LOCAL_RESULTS_DIR)
print('Drive results dir :', RESULTS_DRIVE_DIR)
print('VLM alias         :', VLM_ALIAS)
print('Prompt style      :', PROMPT_STYLE)
print('Retrieval only    :', RETRIEVAL_ONLY)
print('Question limit    :', LIMIT_N_QUESTIONS or 'all')


## 5. Sanity check the gold file

Verifies the file exists and reports the row count. Does not display any question text.

In [ ]:
import json

def _count_rows(path):
    n = 0
    with open(path) as f:
        for line in f:
            if line.strip():
                n += 1
    return n

def _first_row_keys(path):
    with open(path) as f:
        for line in f:
            if line.strip():
                return list(json.loads(line).keys())
    return []

assert Path(GOLD_QA_PATH).exists(), f'gold file not found: {GOLD_QA_PATH}'
print('gold_qa.jsonl rows:', _count_rows(GOLD_QA_PATH))
print('gold record schema:', _first_row_keys(GOLD_QA_PATH))


## 6. Pipeline init (once per session)

In [ ]:
from mrag.config import CFG
CFG.set_vlm_model(VLM_ALIAS)
CFG.set_answer_style(PROMPT_STYLE)
print('VLM model :', getattr(CFG, 'vlm_model_api', '?'))
print('Answer style :', getattr(CFG, 'prompt_style_answer', '?'))

# Try both known locations for init_pipeline.
try:
    from mrag.pipeline import init_pipeline
except ImportError:
    from mrag.ask import init_pipeline

pipeline = init_pipeline()
print('Pipeline ready:', type(pipeline).__name__)
print('Reranker type :', type(getattr(pipeline, 'reranker', None)).__name__)


## 7. Ablations + helper

The `run_and_package(ablation_id)` helper is what each per-ablation cell calls. It:
1. Runs the sweep (150 Qs) with the ablation applied
2. Packages results into `mrag_ablation_<id>.zip` on Drive

Resume-safe: if a sweep was killed midway, re-running skips completed qa_ids.

In [ ]:
from ablations.ablation_shims import ABLATIONS, list_ablations
from ablations.run_ablation import run_ablation_sweep
from ablations.package_results import package_ablation

print('Registered ablations:')
for a in list_ablations():
    print(f'  {a["id"]:20s} — {a["description"]}')


In [ ]:
def run_and_package(ablation_id: str, retrieval_only: bool = None, limit: int = None):
    """Run one ablation's sweep, then zip to Drive. Prints qa_id-level status only."""
    retrieval_only = RETRIEVAL_ONLY if retrieval_only is None else retrieval_only
    limit = LIMIT_N_QUESTIONS if limit is None else limit

    runs_path = LOCAL_RESULTS_DIR / f'runs_{ablation_id}.jsonl'
    zip_path  = RESULTS_DRIVE_DIR / f'mrag_ablation_{ablation_id}.zip'
    print(f'\n=== {ablation_id} ===')
    print(f'    local jsonl : {runs_path}')
    print(f'    drive zip   : {zip_path}')

    counters = run_ablation_sweep(
        ablation_id=ablation_id,
        gold_path=Path(GOLD_QA_PATH),
        output_path=runs_path,
        vlm_alias=VLM_ALIAS,
        prompt_style=PROMPT_STYLE,
        limit=limit,
        retrieval_only=retrieval_only,
    )
    print(f'    sweep counts: {counters}')

    ablation_meta = ABLATIONS.get(ablation_id)
    package_ablation(
        ablation_id=ablation_id,
        runs_path=runs_path,
        zip_output_path=zip_path,
        vlm_alias=VLM_ALIAS,
        prompt_style=PROMPT_STYLE,
        repo_dir=Path(REPO_DIR),
        description=ablation_meta.description if ablation_meta else '',
    )
    print(f'    zip landed  : {zip_path.exists()}')
    return zip_path


## 8. Smoke test (optional — 5 Qs on A1, ~2 min)

Confirms end-to-end wiring before committing to a full 150-Q sweep.

In [ ]:
# Comment this cell out to skip smoke test.
smoke_runs = LOCAL_RESULTS_DIR / 'SMOKE_runs_A1_no_router.jsonl'
if smoke_runs.exists():
    smoke_runs.unlink()

counters = run_ablation_sweep(
    ablation_id='A1_no_router',
    gold_path=Path(GOLD_QA_PATH),
    output_path=smoke_runs,
    vlm_alias=VLM_ALIAS,
    prompt_style=PROMPT_STYLE,
    limit=5,
    retrieval_only=RETRIEVAL_ONLY,
)
print('Smoke result:', counters)
print('Smoke jsonl :', smoke_runs, ' — inspect manually if you like')


## 9. Ablation A1 — no_router

Toggles `CFG.use_question_router = False`. Runs 150 Qs → zips to `mrag_ablation_A1_no_router.zip`.

Re-run only this cell if A1 fails or needs a retry — resume-safe.


In [ ]:
zip_A1 = run_and_package('A1_no_router')


## 10. Ablation A2 — no_vlm_filter

Toggles `CFG.use_vlm_figure_filter = False`. Runs 150 Qs → zips to `mrag_ablation_A2_no_vlm_filter.zip`.


In [ ]:
zip_A2 = run_and_package('A2_no_vlm_filter')


## 11. Ablation A3 — no_graph

Sets `CFG.w_graph = 0.0`. Runs 150 Qs → zips to `mrag_ablation_A3_no_graph.zip`.


In [ ]:
zip_A3 = run_and_package('A3_no_graph')


## 12. Ablation A4 — no_rule_type

Sets `CFG.w_ruletype = 0.0`. Runs 150 Qs → zips to `mrag_ablation_A4_no_rule_type.zip`.


In [ ]:
zip_A4 = run_and_package('A4_no_rule_type')


## 13. Ablation A5 — no_hierarchy

Sets `CFG.w_hierarchy = 0.0`. Runs 150 Qs → zips to `mrag_ablation_A5_no_hierarchy.zip`.


In [ ]:
zip_A5 = run_and_package('A5_no_hierarchy')


## 14. Ablation A6 — no_reranker

Swaps `pipeline.reranker` for `NoOpReranker` at runtime. Runs 150 Qs → zips to `mrag_ablation_A6_no_reranker.zip`.


In [ ]:
zip_A6 = run_and_package('A6_no_reranker')


## 15. Run all remaining (convenience)

Runs whichever of A1–A6 do not yet have a zip on Drive. Skip if you're running cells individually.

In [ ]:
# Idempotent: only runs ablations whose zip is not already on Drive.
ALL_ABLATIONS = ['A1_no_router', 'A2_no_vlm_filter', 'A3_no_graph',
                 'A4_no_rule_type', 'A5_no_hierarchy', 'A6_no_reranker']

produced = []
for aid in ALL_ABLATIONS:
    zpath = RESULTS_DRIVE_DIR / f'mrag_ablation_{aid}.zip'
    if zpath.exists():
        print(f'skip {aid} — zip already exists at {zpath}')
        continue
    produced.append(run_and_package(aid))

print(f'\nDone. Produced {len(produced)} zips this run.')


## 16. Handoff to GPT

The zips on Drive at `RESULTS_DRIVE_DIR` contain raw RAG outputs only. Each
zip has:
- `runs_<id>.jsonl` — one row per question with qa_id, question, answer,
  retrieved chunks, retrieved figures, retrieved pages, retrieval debug, latency
- `meta.json` — ablation config, timestamp, counts, git sha
- `README.txt` — brief note describing the contents

Hand each zip to GPT-5.6 with your MUTCD-150 rubric prompt for scoring.

This harness produced no scores, no aggregations, no interpretation.
